# Cross-model tournament — starting point

Runs trained Octagon models against each other in inference and writes one behavioural-log JSON per matchup (the format `octagon_analysis` consumes). See `tournament/tournament_func.py` and `tournament/Tournament_plan.md`.

**Before running:**
1. Activate the conda env that has `mlagents` (the kernel for this notebook must be that env).
2. Make a **fresh** `TournamentOctagonStage` build (headless Linux `.x86_64`). A fresh build is required so it has `ScriptedTrialSequence.cs` + the timing keys — an older build silently ignores `--trial_seq` and generates random trials. Set its path in `UNITY_ENV_PATH` below.
3. Entrants must be `OctagonAgentSocial` models from the `260715` build or newer (they match the tournament build's 110° FoV observations).

Actions are **sampled** (not greedy), matching training behaviour → matchups are not bit-reproducible; run enough episodes to average over sampling noise. Trial content/timing is pinned by the predetermined `trial_seq`.

In [1]:
import importlib
import sys
from pathlib import Path

# Repo root on sys.path so `tournament.tournament_func` and the
# `trainer_and_simulator_functions` it imports both resolve.
REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Force a reload of both modules. Python caches imports, so editing tournament_func.py has
# NO effect on an already-running kernel. That is how the 260730 batch was lost: the fixed
# file was on disk, but the kernel kept the pre-fix functions in memory, wrote configs with
# no environment_parameters, and froze P2 in all 64 matchups a second time. Reload here,
# and if in any doubt restart the kernel outright.
import trainer_and_simulator_functions as _tsf
importlib.reload(_tsf)
import tournament.tournament_func as tf
importlib.reload(tf)

from tournament.tournament_func import run_tournament, run_matchup, write_tournament_config_yaml

## Config

Edit the paths / picks here. `MODELS_ROOT` is the parent folder; entrants are its immediate subfolders (each with `OctagonAgentSocial/checkpoint.pt`). All entrants must live under one `MODELS_ROOT` — to mix `260715` and `260723`, gather the chosen run dirs under a single folder first.

In [ ]:
OCTAGON_DIR = Path("~/Unity/Octagon").expanduser()

# Fresh TournamentOctagonStage build (headless Linux).
UNITY_ENV_PATH = Path("~/Unity/Octagon/builds/260724/build_260724_01_tournament.x86_64").expanduser()

# Parent folder holding the entrant run dirs (each an immediate subfolder).
MODELS_ROOT = Path("~/Unity/Octagon/results/2607/260715").expanduser()

# Hand-picked entrants (subfolder names under MODELS_ROOT), per tournament_notes_260715.txt.
# Set to None to auto-use EVERY model under MODELS_ROOT (careful: N models => C(N,2)+N matchups).
RUN_IDS = [
    "260715_0022",  # most central (top scorer, player score)
    "260715_0020",  # strong central (2nd scorer)
    "260715_0002",  # 'typical' centre
    "260715_0014",  # intermediate (central leaning)
    "260715_0010",  # intermediate (peripheral leaning) - slower completion tail
    "260715_0036",  # best peripheral scorer + prompt (replaces old 0005)
    "260715_0015",  # peripheral, most High-seeking
    "260715_0026",  # 'typical' peripheral (3e6)
]


# Predetermined trial sequence (pins trial content + timing across all matchups).
# Keep it LONGER than EPISODES — the build stops when the sequence is exhausted.
TRIAL_SEQ = Path("~/repos/agent_training/trial_sequences/trials_100000_seed17.json").expanduser()

# Where the per-matchup JSON logs land: one folder per batch, named for the model batch,
# so a round-robin is self-contained. The earlier 260715 tournaments are void — the
# harness dropped environment_parameters, which froze P2 in all 64 matchups; they now
# live under simulations/tournaments/failed_runs/260729_env_params_bug/ (see
# WHY_THESE_FAILED.txt there). Fixed 260729; this path is a clean re-run.
OUT_ROOT = Path("~/Unity/Octagon/simulations/tournaments/260715").expanduser()

# MLAgents episodes per matchup. Conversion (measured on 260715 inference, episodes=201
# -> ~200 "trial start" events): trials ~= episodes - 1, i.e. ~1 trial per episode.
# So for a target of T trials/matchup, set EPISODES ~= T + 1 (occasional runs finish a
# trial or two short under sampled actions, so treat it as an upper bound).
EPISODES = 350   # ~350 trials/matchup (T+1 per the conversion above)

# Run each distinct pair in BOTH seat orders (mirror matchups). P1/P2 is symmetric
# so this is off in the function by default; ON here to double data (2 x ~250 = ~500
# trials/pair) and to confirm the seat has no effect by comparing a pair vs its mirror.
INCLUDE_MIRRORS = True

# Include models playing against themselves. False by default
INCLUDE_SELF_MATCHUPS = True 

In [3]:
import tempfile

# Fail fast with a clear message if anything is missing.
assert UNITY_ENV_PATH.exists(), f"Build not found: {UNITY_ENV_PATH} (set UNITY_ENV_PATH to your fresh tournament build)"
assert MODELS_ROOT.exists(), f"MODELS_ROOT not found: {MODELS_ROOT}"
assert TRIAL_SEQ.exists(), f"TRIAL_SEQ not found: {TRIAL_SEQ}"
for rid in (RUN_IDS or []):
    ckpts = list((MODELS_ROOT / rid).glob("*/checkpoint.pt"))
    assert ckpts, f"No <behaviour>/checkpoint.pt under {MODELS_ROOT / rid}"

# Stale-kernel guard. Generate a throwaway config and confirm the block is REALLY in it.
# This runs from notebook source, so it catches a cached pre-fix module — which an
# assertion inside tournament_func.py cannot do, because a stale kernel would not have
# that assertion either. Without environment_parameters the build throws on every action,
# CompetitiveAgent2 never gets stepped, and every matchup is a 350-0 walkover that still
# writes a complete, healthy-looking JSON.
with tempfile.TemporaryDirectory() as _tmp:
    _probe = write_tournament_config_yaml(MODELS_ROOT / (RUN_IDS or ["?"])[0], Path(_tmp) / "probe.yaml")
    _txt = _probe.read_text()
assert "environment_parameters" in _txt and "step_penalty" in _txt, (
    "Generated config has NO environment_parameters/step_penalty — the build will throw "
    "every action and freeze P2. Your kernel is running pre-fix code: restart it and "
    "re-run from the top."
)

n = len(RUN_IDS) if RUN_IDS else len(list(MODELS_ROOT.glob('*/*/checkpoint.pt')))
distinct_pairs = n * (n - 1) // 2
n_matchups = distinct_pairs * (2 if INCLUDE_MIRRORS else 1) + n   # + n self-matchups
print(f"All paths OK, config carries step_penalty. {n} entrants => {n_matchups} matchups "
      f"({'with' if INCLUDE_MIRRORS else 'no'} mirrors, incl. self-matchups).")

All paths OK, config carries step_penalty. 8 entrants => 64 matchups (with mirrors, incl. self-matchups).


## (Optional) Single-matchup smoke test

Run one matchup with a few episodes first to confirm the build + checkpoints load and a JSON is produced, before committing to the full round-robin. `run_matchup` needs the CLI config yaml to already exist (the full `run_tournament` writes it for you), so we write it here first.

In [4]:
import collections
import re

config_yaml = write_tournament_config_yaml(
    template_run_dir=MODELS_ROOT / RUN_IDS[0],
    out_yaml=OUT_ROOT / "tournament_config.yaml",
)

log = run_matchup(
    model_a_run_dir=MODELS_ROOT / RUN_IDS[0],
    model_b_run_dir=MODELS_ROOT / RUN_IDS[1],
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_path=OUT_ROOT / "_smoke_test",
    episodes=8,
    config_yaml=config_yaml,
    trial_seq=TRIAL_SEQ,
)
print("smoke-test log:", log)

# Verdict, so the smoke test actually tests something. Both agents must MOVE, the wall
# triggers must be split, and Unity must not be throwing. A frozen opponent still
# produces a complete, full-length JSON and a clean exit — that is exactly how two
# 64-matchup batches were lost.
raw = max((OUT_ROOT / "_smoke_test").glob("*.json"), key=lambda p: p.stat().st_mtime).read_text()
n_trials = raw.count('"trial start"')
uniq = {c: len(set(re.findall(rf'"clientId":{c},"location":\{{[^}}]*\}}', raw))) for c in (0, 1)}
trig = collections.Counter(re.findall(r'"triggerClient":(\d)', raw))
exc_log = (OCTAGON_DIR / "results" / "tournament"
           / f"{RUN_IDS[0]}__vs__{RUN_IDS[1]}" / "run_logs" / "Player-0.log")
n_exc = exc_log.read_text(errors="ignore").count("step_penalty not found") if exc_log.exists() else None

print(f"\ntrials:           {n_trials}")
print(f"unique positions: P1 {uniq[0]}, P2 {uniq[1]}")
print(f"wall triggers:    P1 {trig.get('0', 0)}, P2 {trig.get('1', 0)}")
print(f"step_penalty exceptions in Unity log: {n_exc}")

assert min(uniq.values()) > 1, "AN AGENT IS FROZEN — do not launch the round-robin"
assert n_exc == 0, f"Unity is still throwing on step_penalty ({n_exc}) — do not launch the round-robin"
print("\nSMOKE TEST PASSED — safe to run the full round-robin.")

['mlagents-learn', '/home/tom/Unity/Octagon/simulations/tournaments/260715/tournament_config.yaml', '--run-id', 'tournament/260715_0022__vs__260715_0020', '--resume', '--inference', '--base-port', '5015', '--env', '/home/tom/Unity/Octagon/builds/260724/build_260724_01_tournament.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/tournaments/260715/_smoke_test', '--sim_eps', '8', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
smoke-test log: /home/tom/Unity/Octagon/simulations/tournaments/260715/_smoke_test/mlagents_stdout.log

trials:           8
unique positions: P1 2755, P2 3041
wall triggers:    P1 2, P2 6
step_penalty exceptions in Unity log: 0

SMOKE TEST PASSED — safe to run the full round-robin.


## Full round-robin

`combinations_with_replacement` — every unordered pair, **including self-matchups**. Mirrors (seat-swapped duplicates) are **optional** via `include_mirrors` (default off in the function; set by `INCLUDE_MIRRORS` above). One JSON per matchup under `OUT_ROOT/<A>__vs__<B>/`. Failures are caught per-matchup (that entry is `None`) so one bad run doesn't kill the batch.

**Self-matchups vs mirrors — to be clear:**
- **Self-matchups (a model vs a copy of itself) ARE run** — one per entrant, giving a ~50% baseline / sanity check.
- **Mirrors (the same pairing with the P1/P2 seats swapped) are OFF by default.** Which seat a model takes is irrelevant: agents aren't repositioned at trial start, so each agent's position is emergent from its own behaviour, and the only P1/P2 difference is administrative (arena setup / logging). Set `INCLUDE_MIRRORS = True` to run both seat orders — this doubles data per pair and lets you confirm empirically that the seat has no effect (compare A-vs-B against B-vs-A on the same trial sequence).

In [ ]:
logs = run_tournament(
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    run_ids=RUN_IDS,            # None => all models under MODELS_ROOT
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
    include_mirrors=INCLUDE_MIRRORS,
    trial_seq=TRIAL_SEQ,
    base_port=9100,             # ports = 9100 + 20*i -> 9100..10360. Training uses
                                # 5005+20*i per model, so a session tops ~6400 (70 models)
                                # / ~9000 (200 models); 9100 clears sessions up to ~206
                                # models. Also above tensorboard (6006) / mDNS (5353).
)


Running 64 matchups over 8 models: ['260715_0022', '260715_0020', '260715_0002', '260715_0014', '260715_0010', '260715_0036', '260715_0015', '260715_0026']

=== [1/64] Matchup: 260715_0022 (P1) vs 260715_0022 (P2) ===
['mlagents-learn', '/home/tom/Unity/Octagon/simulations/tournaments/260715/tournament_config.yaml', '--run-id', 'tournament/260715_0022__vs__260715_0022', '--resume', '--inference', '--base-port', '9100', '--env', '/home/tom/Unity/Octagon/builds/260724/build_260724_01_tournament.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/tournaments/260715/260715_0022__vs__260715_0022', '--sim_eps', '350', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
=== Finished: 260715_0022__vs__260715_0022 -> /home/tom/Unity/Octagon/simulations/tournaments/260715/260715_0022__vs__260715_0022/mlagents_stdout.log ===

=== [2/64] Matchup: 260715_0022 (P1) vs 260715_0020 (P2) ===
['mlagents-learn', 

In [ ]:
import collections
import re

# Summary: which matchups produced a log, which failed — and, more importantly, which
# are VALID. A matchup can exit cleanly with a complete 350-trial JSON and still be void:
# before the 260729 fix, a missing environment_parameters block froze P2 at its spawn in
# all 64 matchups and handed P1 every trial, and nothing in the exit status showed it.
ok = {k: v for k, v in logs.items() if v is not None}
failed = [k for k, v in logs.items() if v is None]
print(f"{len(ok)}/{len(logs)} matchups ran without error")
for k in failed:
    print("  FAILED:", k)

print("\nValidity check (both agents must move, both must win some trials):")
suspect = []
for tid in logs:
    jsons = list((OUT_ROOT / tid).glob("*.json"))
    if not jsons:
        print(f"  {tid}: NO JSON")
        suspect.append(tid)
        continue
    raw = max(jsons, key=lambda p: p.stat().st_mtime).read_text()
    n_trials = raw.count('"trial start"')
    uniq = {c: len(set(re.findall(rf'"clientId":{c},"location":\{{[^}}]*\}}', raw))) for c in (0, 1)}
    trig = collections.Counter(re.findall(r'"triggerClient":(\d)', raw))
    bad = min(uniq.values()) <= 1 or min(trig.get('0', 0), trig.get('1', 0)) == 0 or n_trials < EPISODES - 5
    if bad:
        suspect.append(tid)
        print(f"  {tid}: {n_trials} trials | uniq pos P1 {uniq[0]}, P2 {uniq[1]} | "
              f"triggers P1 {trig.get('0', 0)}, P2 {trig.get('1', 0)}  <-- SUSPECT")

print(f"\n{len(logs) - len(suspect)}/{len(logs)} matchups look valid."
      + ("" if not suspect else f" Investigate before analysing: {suspect}"))

## Rerun specific matchups

For matchups that failed or came out short in the round-robin above, without re-running the whole batch. List the ordered `(P1, P2)` pairs in `RERUN` — the same ids that make up the matchup dir name `<P1>__vs__<P2>` — and set the overrides below.

Everything else (build, checkpoints, trial sequence, sampled actions) is identical to the round-robin, so the reruns are drop-in replacements for the originals.

**Note on `260715_0002__vs__260715_0002`:** it did not crash — it stalled. Its stdout log is full of `No episode was completed since last summary` and it produced only **68** `trial start` events in ~7200 s before hitting the 5000 s `timeout_s`, against ~350 trials in ~2200 s for a healthy matchup. So this pairing deadlocks: both copies of 0002 wander without triggering a wall. Actions are sampled, so a **different `RERUN_SEED` re-rolls the trajectories** and is the thing most likely to change the outcome; raising `RERUN_TIMEOUT_S` alone will mostly just buy a longer stall. Check the trial count printed at the end, not just "no exception".

In [ ]:
import collections
import re
import shutil
import time

# Ordered (P1, P2) run_id pairs to rerun. Same id twice => self-matchup.
RERUN = [
    ("260715_0002", "260715_0002"),
]

# Folder holding this batch's matchup dirs — the same OUT_ROOT the round-robin wrote to,
# so the rerun lands beside its siblings.
RERUN_OUT_ROOT = OUT_ROOT

RERUN_EPISODES  = EPISODES   # trials ~= episodes - 1
RERUN_SEED      = 17         # change to re-roll the sampled trajectories
RERUN_TIMEOUT_S = 5000       # wall-clock cap per matchup (healthy 350-ep run: ~2200 s)
RERUN_BASE_PORT = 10500      # clear of the round-robin's 9100..10360
ARCHIVE_EXISTING = True      # move the old matchup dir to OUT_ROOT/../superseded/ first,
                             # so the rerun's JSON is the only one in the dir

assert all((MODELS_ROOT / m).exists() for pair in RERUN for m in pair), "unknown run_id in RERUN"

rerun_config_yaml = write_tournament_config_yaml(
    template_run_dir=MODELS_ROOT / RERUN[0][0],
    out_yaml=RERUN_OUT_ROOT / "tournament_config.yaml",
)

rerun_logs = {}
for i, (a, b) in enumerate(RERUN):
    tournament_id = f"{a}__vs__{b}"
    out_path = RERUN_OUT_ROOT / tournament_id

    if ARCHIVE_EXISTING and out_path.exists():
        archive_to = (RERUN_OUT_ROOT.parent / "superseded"
                      / f"{tournament_id}__{time.strftime('%Y%m%d_%H%M%S')}")
        archive_to.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(out_path), str(archive_to))
        print(f"archived previous output -> {archive_to}")

    print(f"\n=== [{i+1}/{len(RERUN)}] Rerun: {a} (P1) vs {b} (P2) ===")
    try:
        log = run_matchup(
            model_a_run_dir=MODELS_ROOT / a,
            model_b_run_dir=MODELS_ROOT / b,
            octagon_dir=OCTAGON_DIR,
            unity_env_path=UNITY_ENV_PATH,
            out_path=out_path,
            episodes=RERUN_EPISODES,
            config_yaml=rerun_config_yaml,
            tournament_id=tournament_id,
            base_port=RERUN_BASE_PORT + 20 * i,
            timeout_s=RERUN_TIMEOUT_S,
            seed=RERUN_SEED,
            trial_seq=TRIAL_SEQ,
        )
        rerun_logs[tournament_id] = log
        print(f"=== Finished: {tournament_id} -> {log} ===")
    except Exception as e:
        rerun_logs[tournament_id] = None
        print(f"=== FAILED: {tournament_id} — {type(e).__name__}: {e} ===")

# Validity check, not just "no exception". A run can produce a complete, full-length JSON
# and still be void: the 260729 env-params bug froze P2 at its spawn and gave P1 every
# trial in all 64 matchups. Both players must MOVE and both must win some trials.
print()
for tid in rerun_logs:
    jsons = list((RERUN_OUT_ROOT / tid).glob("*.json"))
    if not jsons:
        print(f"{tid}: NO JSON WRITTEN")
        continue
    newest = max(jsons, key=lambda p: p.stat().st_mtime)
    raw = newest.read_text()
    n_trials = raw.count('"trial start"')
    uniq = {c: len(set(re.findall(rf'"clientId":{c},"location":\{{[^}}]*\}}', raw))) for c in (0, 1)}
    trig = collections.Counter(re.findall(r'"triggerClient":(\d)', raw))
    flags = []
    if n_trials < RERUN_EPISODES - 5:
        flags.append(f"SHORT (expected ~{RERUN_EPISODES - 1})")
    if min(uniq.values()) <= 1:
        flags.append("AGENT FROZEN — matchup is void")
    if min(trig.get('0', 0), trig.get('1', 0)) == 0:
        flags.append("one player won every trial")
    print(f"{tid}: {n_trials} trials | unique positions P1 {uniq[0]}, P2 {uniq[1]} | "
          f"triggers P1 {trig.get('0', 0)}, P2 {trig.get('1', 0)}"
          + ("  <-- " + "; ".join(flags) if flags else "  OK"))